In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
from scipy import signal
import scipy.io as sio
import difflib
from collections import Counter
from scipy.linalg import toeplitz
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import f1_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from xgboost import XGBClassifier

2026-07-13 15:41:57.960336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783957318.271206      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783957318.359621      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783957319.087863      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783957319.087933      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783957319.087936      57 computation_placer.cc:177] computation placer alr

In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
import os

file_path = '/kaggle/input/datasets/ummerummanchaity/fors-emg-a-novel-semg-dataset/FORS-EMG Dataset/FORS-EMG Dataset/FORS-EMG/Subject14/Pronation/Hand_Open-2.mat'

if os.path.exists(file_path):
    mat = sio.loadmat(file_path)
    keys = list(mat.keys())
    print("Keys in MAT file:", keys)
    data_key = keys[-1]  # Assuming the last key is the data matrix
    emg_data = mat[data_key]
    print("EMG data shape:", emg_data.shape)
    
    # Assuming data is samples x channels, transpose if necessary
    if emg_data.shape[0] == 8000:
        emg_data = emg_data.T  # To channels x samples
    
    sample_rate = 985
    num_channels = emg_data.shape[0]
    num_samples = emg_data.shape[1]
    time = np.arange(num_samples) / sample_rate
    
    fig, axs = plt.subplots(num_channels, 1, figsize=(12, 2*num_channels), sharex=True)
    fig.suptitle('sEMG Signals for Hand_Close Gesture (Pronation, Subject1, Trial1)')
    
    for ch in range(num_channels):
        axs[ch].plot(time, emg_data[ch], label=f'Channel {ch+1}')
        axs[ch].set_ylabel('Amplitude')
        axs[ch].legend(loc='upper right')
        axs[ch].grid(True)
    
    axs[-1].set_xlabel('Time (seconds)')
    
    # Instead of plt.show(), save to a temp file and print description
    plt.savefig('emg_plot.png')
    print("Plot saved as 'emg_plot.png'. Description: Multi-channel sEMG plot with time on x-axis and amplitude on y-axis for each channel.")
else:
    print("File not found at:", file_path)

In [ ]:
BASE_PATH = ('/kaggle/input/datasets/ummerummanchaity/fors-emg-a-novel-semg-dataset/'
             'FORS-EMG Dataset/FORS-EMG Dataset/FORS-EMG')

N_FOLDS = 5
RANDOM_SEED = 200

sample_rate = 985
window_size_ms = 250
stride_ms = 250
window_size = int(window_size_ms / 1000 * sample_rate)
stride = int(stride_ms / 1000 * sample_rate)
num_channels = 8
notch_freq = 50.0 
q = 30.0

ONSET_ENV_MS = 150
ONSET_SUSTAIN_MS = 200
ONSET_FRAC = 0.2
ONSET_HI_PCT = 90
FALLBACK_ONSET_S = 0.5 

zc_threshold = 0.0
ssc_threshold = 0.0

gestures = ['Thumb_Up', 'Index', 'Right_Angle', 'Peace', 'Index_Little', 'Thumb_Little', 'Hand_Close', 'Hand_Open', 'Wrist_Extension', 'Wrist_Flexion', 'Ulnar_Deviation', 'Radial_Deviation']
gesture_map = {g: i for i, g in enumerate(gestures)}
orientations = ['pronation', 'rest', 'supination']
orientation_map = {o: i for i, o in enumerate(orientations)}

upper_orientations = ['Pronation', 'Rest', 'Supination']
upper_orientation_map = {o: i for i, o in enumerate(upper_orientations)}

SUBJECTS = [3]
FEATURES = 'combine'

In [ ]:
def _detect_onset_s(emg, fs=sample_rate, env_ms=ONSET_ENV_MS, sustain_ms=ONSET_SUSTAIN_MS, frac=ONSET_FRAC, hi_pct=ONSET_HI_PCT):

    x = emg.astype(float)
    x = x - x.mean(axis=1, keepdims=True)              
    w = max(1, int(env_ms / 1000 * fs))
    if x.shape[1] < w:
        return None
    env = np.zeros(x.shape[1] - w + 1)
    for ch in x:
        env += np.sqrt(np.convolve(ch ** 2, np.ones(w) / w, mode='valid'))
    env /= x.shape[0]
 
    lo, hi = np.percentile(env, 5), np.percentile(env, hi_pct)   # hi robust (không dùng max)
    if hi - lo < 1e-9:
        return None
    envn = np.clip((env - lo) / (hi - lo), 0, None)
    sus = max(1, int(sustain_ms / 1000 * fs))
    above = envn > frac
    for i in range(len(above) - sus):
        if above[i:i + sus].all():
            return (i + w // 2) / fs
    return None

def crop_active(emg_trial, K=5.0, fs=sample_rate, fallback_onset_s=FALLBACK_ONSET_S, return_info=False):
    
    emg_trial = np.asarray(emg_trial)
    N = emg_trial.shape[1]
    win = int(round(K * fs))
 
    if N <= win:
        info = dict(onset_s=0.0, start=0, end=N, used_fallback=True, clamped=True, reason='signal<=win')
        return (emg_trial, info) if return_info else emg_trial
 
    onset_s = _detect_onset_s(emg_trial, fs=fs)
    used_fallback = onset_s is None
    if used_fallback:
        onset_s = fallback_onset_s
 
    start = int(round(onset_s * fs))
    end = start + win
    clamped = False
    if end > N:                          
        end = N
        start = N - win
        clamped = True
    if start < 0:                       
        start, end = 0, win
 
    crop = emg_trial[:, start:end]
    if return_info:
        return crop, dict(onset_s=round(onset_s, 3), start=start, end=end, dur_s=round((end - start) / fs, 3), used_fallback=used_fallback, clamped=clamped)
        
    return crop

def generate_folds_per_orientation(y_gesture, y_orientation, n_folds=N_FOLDS, seed=RANDOM_SEED):

    y_gesture = np.asarray(y_gesture)
    y_orientation = np.asarray(y_orientation)
    all_trial_idx = np.arange(len(y_gesture))

    fold_results = {}
    for ori in np.unique(y_orientation):
        ori_mask = (y_orientation == ori)
        ori_trial_idx = all_trial_idx[ori_mask]   # global trial indices for this orientation
        ori_gestures = y_gesture[ori_mask]        # gesture labels within this orientation
 
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        fold_results[ori] = {}
        # split() needs an X of the right length; labels drive the stratification
        for fold, (tr, te) in enumerate(skf.split(np.zeros(len(ori_gestures)), ori_gestures)):
            fold_results[ori][fold] = {
                'train_trial_idx': ori_trial_idx[tr],
                'test_trial_idx':  ori_trial_idx[te],
            }
    return fold_results

def print_fold_balance_check(fold_results, y_gesture, y_orientation, fold_num=0):
    y_gesture = np.asarray(y_gesture)
    print(f'\n=== Fold {fold_num} balance (per orientation) ===')
    for ori in sorted(fold_results.keys()):
        train = fold_results[ori][fold_num]['train_trial_idx']
        test = fold_results[ori][fold_num]['test_trial_idx']
        train_g = Counter(y_gesture[train]); test_g = Counter(y_gesture[test])
        print(f'  ori {ori}: train={len(train)} {dict(sorted(train_g.items()))} | '
              f'test={len(test)} {dict(sorted(test_g.items()))}')

def merge_orientations(fold_results, fold_num, orientations_to_use=None):
    if orientations_to_use is None:
        orientations_to_use = sorted(fold_results.keys())
    train_idx, test_idx = [], []
    for ori in orientations_to_use:
        train_idx.extend(fold_results[ori][fold_num]['train_trial_idx'].tolist())
        test_idx.extend(fold_results[ori][fold_num]['test_trial_idx'].tolist())
    return np.array(train_idx), np.array(test_idx)

def build_window_dataset(X_list, y_gesture, y_orientation, trial_idx_list, features):
    feats_list, g_labels, o_labels = [], [], []
    for ti in trial_idx_list:
        feat = process_trial(X_list[ti], features=features)   # [n_windows, F]
        if feat.ndim != 2 or feat.shape[0] == 0:
            print(f'[WARN] trial {ti} produced no windows — skipped')
            continue
        n_windows = feat.shape[0]
        feats_list.append(feat)
        g_labels.extend([y_gesture[ti]] * n_windows)
        o_labels.extend([y_orientation[ti]] * n_windows)
    X = np.vstack(feats_list)
    return X, np.array(g_labels), np.array(o_labels)

def ensure_channels_first(emg, n_ch=num_channels):
    emg = np.asarray(emg)
    if emg.ndim != 2:
        raise ValueError(f"EMG không phải 2D: shape {emg.shape}")
    if emg.shape[0] == n_ch:
        return emg
    if emg.shape[1] == n_ch:
        return emg.T
    raise ValueError(f"Không trục nào = {n_ch} kênh: shape {emg.shape}")

def load_data(base_path='/kaggle/input/datasets/ummerummanchaity/fors-emg-a-novel-semg-dataset/FORS-EMG Dataset/FORS-EMG Dataset/FORS-EMG', subjects=None):
    if subjects is None:
        subjects = range(1, 20)
    X_list = []
    y_gesture_list = []
    y_orientation_list = []

    for subject in subjects:
        subject_path = os.path.join(base_path, f'Subject{subject}')
        if subject != 1:
            new_orientations = ['Pronation', 'Rest', 'Supination']
        else:
            new_orientations = orientations
        for ori in new_orientations:
            ori_path = os.path.join(subject_path, ori)
            for gest in gestures:
                for trial in range(1, 6):
                    if gest == 'Thumb_Up':
                        rename_gest = 'Thumb_UP'
                    elif gest == 'Ulnar_Deviation':
                        rename_gest = 'Ulner_Deviation'
                    else:
                        rename_gest = gest
                    file_name = f'{rename_gest}-{trial}.mat'
                    file_path = os.path.join(ori_path, file_name)
                    mat = sio.loadmat(file_path)
                    emg_data = mat[list(mat.keys())[-1]]
                    emg_trial = ensure_channels_first(emg_data) 
                    emg_trial = crop_active(emg_trial)
                    X_list.append(emg_trial)
                    y_gesture_list.append(gesture_map[gest])
                    if subject != 1:
                        y_orientation_list.append(upper_orientation_map[ori])
                    else:
                        y_orientation_list.append(orientation_map[ori])

    #print("Load data function")
    #print(f"[DEBUG] Loaded {len(X_list)} EMG trials")
    #for i, seg in enumerate(X_list[:5]):  # only preview first 5
    #    print(f"   Trial {i} shape: {seg.shape}")
        
    return X_list, np.array(y_gesture_list), np.array(y_orientation_list)

def signal_normalize_window(window, eps=1e-8):
    return window / np.sqrt(np.mean(window ** 2) + eps)
            
def apply_bandpass(data, fs=sample_rate, low=20, high=450):
    sos = signal.butter(4, [low, high], btype='band', fs=fs, output='sos')
    filtered = signal.sosfiltfilt(sos, data, axis=1)
    return filtered

def apply_notch_filter(data, fs=sample_rate, freq=notch_freq, q=q):
    b, a = signal.iirnotch(freq, q, fs)
    filtered = signal.filtfilt(b, a, data, axis=1)
    return filtered

#def offset_correction(data):
#    return data - np.mean(data, axis=1, keepdims=True)

def extract_windows(data):
    num_samples = data.shape[1]
    windows = []
    for start in range(0, num_samples - window_size + 1, stride):
        window = data[:, start:start + window_size]
        windows.append(window)
    return np.array(windows)

def extract_hudgins_9_features(window):
    if window.ndim != 2:
        raise ValueError("Window must be a 2D array (channels × samples)")
        
    mean_val = np.mean(np.abs(window), axis=1) 
    zc = np.sum(
        (np.diff(np.sign(window), axis=1) != 0)
        & (np.abs(np.diff(window, axis=1)) >= zc_threshold),
        axis=1,
    )
    
    mid   = window[:, 1:-1]
    left  = window[:, :-2]
    right = window[:, 2:]
    ssc = np.sum(((mid - left) * (mid - right)) > ssc_threshold, axis=1)
    
    wl = np.sum(np.abs(np.diff(window, axis=1)), axis=1)

    rms = np.sqrt(np.mean(window**2, axis=1))
    var = np.var(window, axis=1)
    std = np.std(window, axis=1)
    mean_w = np.mean(window, axis=1, keepdims=True)
    std_w = np.where(std < 1e-8, 1e-8, std)[:, None]
    kurt = np.nan_to_num(np.mean(((window - mean_w) / std_w) ** 4, axis=1) - 3.0, nan=0.0)   
    skew = np.nan_to_num(np.mean(((window - mean_w) / std_w) ** 3, axis=1), nan=0.0)

    features = np.concatenate([mean_val, zc, ssc, wl, rms, var, std, kurt, skew])
    return features

def extract_hudgins_features(window):
    if window.ndim != 2:
        raise ValueError("Window must be a 2D array (channels × samples)")
        
    mean_val = np.mean(np.abs(window), axis=1) 
    zc = np.sum(
        (np.diff(np.sign(window), axis=1) != 0)
        & (np.abs(np.diff(window, axis=1)) >= zc_threshold),
        axis=1,
    )
    
    mid   = window[:, 1:-1]
    left  = window[:, :-2]
    right = window[:, 2:]
    ssc = np.sum(((mid - left) * (mid - right)) > ssc_threshold, axis=1)

    wl = np.sum(np.abs(np.diff(window, axis=1)), axis=1)

    features = np.concatenate([mean_val, zc, ssc, wl])
    return features

def extract_sntdf_features(window, eps=1e-8):
        
    d1 = np.diff(window, axis=1)
    d2 = np.diff(window, n=2, axis=1)
    d3 = np.diff(window, n=3, axis=1) 

    mean_val = np.log(np.mean(np.abs(window), axis=1) + eps)
    p0 = np.log(np.sum(window**2, axis=1) + eps)
    p2 = np.log(np.sum(d1**2, axis=1) + eps)
    p4 = np.log(np.sum(d2**2, axis=1) + eps)
    p6 = np.log(np.sum(d3**2, axis=1) + eps)

    ac1 = np.log(np.mean(np.abs(d1), axis=1) + eps)
    ac2 = np.log(np.mean(np.abs(d2), axis=1) + eps)

    corr = np.corrcoef(window)
    corr = np.nan_to_num(corr, nan=0.0) 
    corr_flat = corr[np.triu_indices(num_channels, k=1)] # Upper triangle

    features = np.concatenate([mean_val, p0, p2, p4, p6, ac1, ac2, corr_flat])

    return features
  
def process_trial(emg_trial, features='hudgin'):
    #print('to process trial')
    bandpassed = apply_bandpass(emg_trial)
    filtered = apply_notch_filter(bandpassed)
    windows = extract_windows(filtered)
    #corrected = offset_correction(filtered)
    #windows = extract_windows(corrected)
    feats = []
    for w in windows:
        if features=='hudgin':
            feats.append(extract_hudgins_features(w))
        elif features=='hudgin_9':
            feats.append(extract_hudgins_9_features(w))
        elif features == 'sntdf':
            feats.append(extract_sntdf_features(signal_normalize_window(w)))
        elif features == 'combine':
            feats.append(np.concatenate([
                extract_sntdf_features(signal_normalize_window(w)),        
                extract_hudgins_9_features(w),       
            ]))
        else:
            raise ValueError(f"Unsupported features: {features}")
            
    return np.array(feats)

def normalize_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_norm = scaler.fit_transform(X_train)
    X_test_norm = scaler.transform(X_test)
    return X_train_norm, X_test_norm

class MLP48(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim=None, n_classes=None, lr=1e-3, epochs=30, batch_size=32, verbose=0):
        self.input_dim = input_dim
        self.n_classes = n_classes
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose
        self.model = None
        self.loss_history = []

    def _build_model(self):
        model = Sequential()
        model.add(Dense(48, activation='relu', input_dim=self.input_dim))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
        model.add(Dense(24, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
        model.add(Dense(self.n_classes, activation='softmax'))
        model.compile(optimizer=Adam(learning_rate=self.lr),
                      loss='categorical_crossentropy', metrics=['accuracy'])
        return model

    def fit(self, X, y):
        if self.input_dim is None:
            self.input_dim = X.shape[1]
        if self.n_classes is None:
            self.n_classes = len(np.unique(y))
        y_cat = to_categorical(y, num_classes=self.n_classes)
        self.model = self._build_model()
        history = self.model.fit(X, y_cat, epochs=self.epochs,
                                 batch_size=self.batch_size, verbose=self.verbose)
        self.loss_history = history.history['loss']
        return self

    def predict(self, X):
        return np.argmax(self.model.predict(X, verbose=0), axis=1)


def meta_learner_ensemble(X_train, y_train, X_test, random_state=42, cv=4):
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    n_cls = len(np.unique(y_train_enc))

    svc = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=random_state)
    xgb = XGBClassifier(
        eval_metric='mlogloss', n_estimators=100, max_depth=4,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        random_state=random_state, verbosity=0
    )
    mlp48 = MLP48(input_dim=X_train.shape[1], n_classes=n_cls, epochs=30, verbose=0)
    base_models = [svc, xgb, mlp48]

    cv_split = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    oof_probas  = np.zeros((X_train.shape[0], n_cls * len(base_models)))
    test_probas = np.zeros((X_test.shape[0],  n_cls * len(base_models)))

    for i, model in enumerate(base_models):
        oof_fold_preds  = np.zeros((X_train.shape[0], n_cls))
        test_fold_preds = np.zeros((cv, X_test.shape[0], n_cls))

        for j, (tr_idx, val_idx) in enumerate(cv_split.split(X_train, y_train_enc)):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr        = y_train_enc[tr_idx]

            if isinstance(model, MLP48):
                tf.random.set_seed(random_state + j)
                m = MLP48(input_dim=X_train.shape[1], n_classes=n_cls, epochs=30, verbose=0)
                m.fit(X_tr, y_tr)
                val_pred  = m.model.predict(X_val,  verbose=0)
                test_pred = m.model.predict(X_test, verbose=0)
            else:
                m = model.__class__(**model.get_params())
                m.fit(X_tr, y_tr)
                val_pred  = m.predict_proba(X_val)
                test_pred = m.predict_proba(X_test)

            oof_fold_preds[val_idx] = val_pred
            test_fold_preds[j]      = test_pred

        oof_probas[:,  i*n_cls:(i+1)*n_cls] = oof_fold_preds
        test_probas[:, i*n_cls:(i+1)*n_cls] = test_fold_preds.mean(axis=0)

    scaler = MinMaxScaler()
    X_meta_train = scaler.fit_transform(oof_probas)
    X_meta_test  = scaler.transform(test_probas)

    meta_model = LogisticRegression(max_iter=500, random_state=random_state)
    meta_model.fit(X_meta_train, y_train_enc)

    y_pred_train = le.inverse_transform(meta_model.predict(X_meta_train))
    y_pred_test  = le.inverse_transform(meta_model.predict(X_meta_test))
    return y_pred_train, y_pred_test

def hmc_classification(X_train, X_test, yg_train, yg_test, yo_train, yo_test):

    X_train_norm = np.asarray(X_train, dtype=np.float64)
    X_test_norm  = np.asarray(X_test,  dtype=np.float64)
    X_train_norm, X_test_norm = normalize_data(X_train_norm, X_test_norm)

    ori_pred_train, ori_pred_test = meta_learner_ensemble(
        X_train_norm, yo_train, X_test_norm,
        random_state=RANDOM_SEED, cv=4
    )
    ori_acc = accuracy_score(yo_test, ori_pred_test)

    ori_pred_norm_test  = (ori_pred_test  - np.min(ori_pred_test))  / (np.max(ori_pred_test)  - np.min(ori_pred_test)  + 1e-8)
    ori_pred_norm_train = (ori_pred_train - np.min(ori_pred_train)) / (np.max(ori_pred_train) - np.min(ori_pred_train) + 1e-8)
    X_aug_train = np.hstack((X_train_norm, ori_pred_norm_train.reshape(-1, 1)))
    X_aug_test  = np.hstack((X_test_norm,  ori_pred_norm_test.reshape(-1, 1)))

    gesture_clfs = {}
    for ori in np.unique(yo_train):
        mask = (yo_train == ori)
        if len(np.unique(yg_train[mask])) > 1:
            clf_g = LinearDiscriminantAnalysis()
            clf_g.fit(X_aug_train[mask], yg_train[mask])
            gesture_clfs[ori] = clf_g

    preds_gesture = []
    for i in range(len(X_aug_test)):
        o_pred = ori_pred_test[i]
        x_aug  = X_aug_test[i].reshape(1, -1)
        preds_gesture.append(gesture_clfs[o_pred].predict(x_aug)[0])

    preds_gesture = np.array(preds_gesture)
    multi_label_acc = np.mean((ori_pred_test == yo_test) & (preds_gesture == yg_test))
    soft_acc = accuracy_score(yg_test, preds_gesture)
    return ori_acc, multi_label_acc, soft_acc

In [ ]:
if __name__ == '__main__':
 
    all_means = []
    for subj in SUBJECTS:
        # trial-level data (each X_list[i] is one trial: channels x samples)
        X_list, y_gesture, y_orientation = load_data(subjects=[subj])
        print(f'\n##### Subject {subj} #####')
        print(f'Loaded {len(X_list)} trials | gestures={np.unique(y_gesture)} '
              f'| orientations={np.unique(y_orientation)}')
 
        fold_results = generate_folds_per_orientation(
            y_gesture, y_orientation, n_folds=N_FOLDS, seed=RANDOM_SEED)
        print_fold_balance_check(fold_results, y_gesture, y_orientation, fold_num=0)
 
        fold_metrics = []
        for fold in range(N_FOLDS):
            train_idx, test_idx = merge_orientations(fold_results, fold)
            overlap = set(train_idx) & set(test_idx)
            assert len(overlap) == 0, f'TRIAL LEAKAGE in fold {fold}: {len(overlap)}'
 
            X_train, yg_train, yo_train = build_window_dataset(
                X_list, y_gesture, y_orientation, train_idx, FEATURES)
            X_test, yg_test, yo_test = build_window_dataset(
                X_list, y_gesture, y_orientation, test_idx, FEATURES)
 
            # guard the HMC precondition: every test orientation must exist in train
            missing = set(np.unique(yo_test)) - set(np.unique(yo_train))
            if missing:
                print(f'  [!] orientations {missing} in TEST but not TRAIN (fold {fold})')
 
            ori_acc, ml_acc, soft_acc = hmc_classification(
                X_train, X_test, yg_train, yg_test, yo_train, yo_test)
            print(f'Fold {fold}: train_win={X_train.shape[0]} test_win={X_test.shape[0]} | '
                  f'OriAcc={ori_acc:.4f} MultiLabel={ml_acc:.4f} Soft={soft_acc:.4f}')
            fold_metrics.append((ori_acc, ml_acc, soft_acc))
 
        fold_metrics = np.array(fold_metrics)
        means = fold_metrics.mean(axis=0)
        print('\n=== MEAN over 5 trial-level folds ===')
        print(f'  Orientation Acc : {means[0]:.4f}')
        print(f'  Multi-label Acc : {means[1]:.4f}')
        print(f'  Soft Acc        : {means[2]:.4f}')
        all_means.append(means)
 
    all_means = np.array(all_means)
    print('\n=== Across subjects ===')
    print('  mean Orientation Acc:', np.mean(all_means[:, 0]))
    print('  mean Multi-label Acc:', np.mean(all_means[:, 1]))
    print('  mean Soft Acc       :', np.mean(all_means[:, 2]))